# Procesamiento de datos

In [10]:
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from gensim.models import Word2Vec, KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize, word_tokenize
from transformers import BertTokenizer, BertModel
import torch.nn.functional as F
import torch
import nltk
import numpy as np
from gensim.models.callbacks import CallbackAny2Vec

# Descargar datasets y combinarlos

In [11]:

# Diccionario para mapear tipos de falacia del segundo dataset a las clases del primero
fallacy_mapping = {
    # ad hominem
    "Ad Hominem": "ad hominem",
    "Circumstantial Ad Hominem": "ad hominem",
    "Tu Quoque": "ad hominem",
    "Abusive Ad Hominem": "ad hominem",
    "Guilt By Association": "ad hominem",
    "Argument From Commitment": "ad hominem",
    "Precedent Ad Hominem": "ad hominem",
    "Behavioral Ad Hominem": "ad hominem",
    "Ad Hominem Against a Witness at Trial": "ad hominem",

    # false dilemma
    "False Dichotomy": "false dilemma",
    "False Dilemma/Dichotomy": "false dilemma",
    "False dilemma": "false dilemma",

    # ad populum
    "Appeal to Popularity": "ad populum",
    "Bandwagon Fallacy": "ad populum",
    "Common Belief Fallacy": "ad populum",

    # equivocation
    "Equivocation": "equivocation",

    # fallacy of credibility
    "Argument from Authority": "fallacy of credibility",
    "Appeal to Authority": "fallacy of credibility",
    "Appeal to False Authority": "fallacy of credibility",
    "Argument from False Authority": "fallacy of credibility",
    "Appealing to an irrelevant authority": "fallacy of credibility",

    # false causality
    "Correlation does not imply causation": "false causality",
    "False cause": "false causality",
    "Post hoc ergo propter hoc": "false causality",
    "Cum hoc ergo propter hoc": "false causality",

    # intentional
    "Intentional Fallacy": "intentional",
    "Authorial Intent as Constraint": "intentional",

    # fallacy of logic / circular reasoning
    "Circular Reasoning": "circular reasoning",
    "Circular reasoning": "circular reasoning",
    "Fallacy of Logic": "fallacy of logic",
    "Begging the question": "fallacy of logic",
    "Begging the Question": "fallacy of logic",

    # appeal to emotion
    "Appeal to Emotion": "appeal to emotion",
    "Appeal to emotion": "appeal to emotion",
    "Appeal to Pity": "appeal to emotion",
    "Appeal to fear": "appeal to emotion",
    "Appeal to consequences": "appeal to emotion",

    # fallacy of relevance / extension
    "Fallacy of Extension": "fallacy of extension",
    "Fallacy of Relevance": "fallacy of relevance",
    "Red Herring": "fallacy of relevance",
    "Straw Man": "fallacy of relevance",
    "Straw man": "fallacy of relevance",
    "Strawman": "fallacy of relevance",

    # faulty generalization
    "Hasty Generalization": "faulty generalization",
    "Faulty Generalization ": "faulty generalization",
    "Hasty generalization": "faulty generalization",
    "Accident": "faulty generalization",
    "Generalization": "faulty generalization",
}

# Cargar datasets
dataset1 = load_dataset("tasksource/logical-fallacy")
dataset2 = load_dataset("MrOvkill/fallacies-fallacy-base")

train1, test1, dev1 = dataset1["train"], dataset1["test"], dataset1["dev"]

# Mapear clases del dataset2 a las del dataset1
def map_fallacy(example):
    mapped = fallacy_mapping.get(example["name"])
    return {"logical_fallacies": mapped}

dataset2_mapped = dataset2["train"].map(map_fallacy)
dataset2_mapped = dataset2_mapped.filter(lambda x: x["logical_fallacies"] is not None)

# Mantener solo columnas necesarias
dataset2_mapped = dataset2_mapped.remove_columns(
    [c for c in dataset2_mapped.column_names if c not in ["logical_fallacies", "example"]]
)

# Dividir dataset2 en train/dev/test (80/10/10)
data_array = dataset2_mapped["example"]
labels_array = dataset2_mapped["logical_fallacies"]

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    data_array, labels_array, test_size=0.2, stratify=labels_array, random_state=42
)
dev_texts, test_texts, dev_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

train2 = Dataset.from_dict({"example": train_texts, "logical_fallacies": train_labels})
dev2   = Dataset.from_dict({"example": dev_texts, "logical_fallacies": dev_labels})
test2  = Dataset.from_dict({"example": test_texts, "logical_fallacies": test_labels})

train2 = train2.rename_column("example", "source_article")
dev2   = dev2.rename_column("example", "source_article")
test2  = test2.rename_column("example", "source_article")

# Combinar datasets
train_combined = concatenate_datasets([train1, train2])
dev_combined   = concatenate_datasets([dev1, dev2])
test_combined  = concatenate_datasets([test1, test2])

dataset = DatasetDict({
    "train": train_combined,
    "dev": dev_combined,
    "test": test_combined
})

train = dataset["train"]
dev   = dataset["dev"]
test  = dataset["test"]


print(dataset)
print(f"Train examples: {len(dataset['train'])}")
print(f"Dev examples:   {len(dataset['dev'])}")
print(f"Test examples:  {len(dataset['test'])}")


DatasetDict({
    train: Dataset({
        features: ['config', 'source_article', 'logical_fallacies'],
        num_rows: 2901
    })
    dev: Dataset({
        features: ['config', 'source_article', 'logical_fallacies'],
        num_rows: 598
    })
    test: Dataset({
        features: ['config', 'source_article', 'logical_fallacies'],
        num_rows: 539
    })
})
Train examples: 2901
Dev examples:   598
Test examples:  539


In [12]:
# Descargar modelos de tonekizacion
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\marco\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\marco\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Tokenizacion + case-folding

In [13]:

def tokenize_flat(text):
    text = text.lower()
    sentences = sent_tokenize(text)
    tokens = []
    for sentence in sentences:
        tokens.extend(word_tokenize(sentence))
    return tokens

# NOTE: Esta funcion tokeniza cada frase por separado. Podria ser interesante.
def preprocess_sentences(text):
    text = text.lower()
    sentences = sent_tokenize(text)
    tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]
    return tokenized_sentences

train = train.map(lambda x: {"tokenized": tokenize_flat(x["source_article"])})
test  = test.map(lambda x: {"tokenized": tokenize_flat(x["source_article"])})
dev   = dev.map(lambda x: {"tokenized": tokenize_flat(x["source_article"])})

print(train[1]["source_article"])
print(train[1]["tokenized"])


The bigger a child's shoe size, the better the child's handwriting
['the', 'bigger', 'a', 'child', "'s", 'shoe', 'size', ',', 'the', 'better', 'the', 'child', "'s", 'handwriting']


# Bag of Words

In [14]:

train_corpus = [" ".join(tokens) for tokens in train["tokenized"]]

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(train_corpus)

print("==== Bag-of-Words ====")
print(f"Dimensiones de la matriz BoW: {bow_matrix.shape}")
print(f"Ejemplo de vocabulario:       {list(bow_vectorizer.get_feature_names_out()[1000:1016])}")

# Calculamos similitud entre algunos documentos
similarity_bow = cosine_similarity(bow_matrix[:5])
print("\nSimilitud coseno entre primeros documentos (BoW):")
print(np.round(similarity_bow, 3))


==== Bag-of-Words ====
Dimensiones de la matriz BoW: (2901, 8156)
Ejemplo de vocabulario:       ['boast', 'boat', 'boats', 'bob', 'bobby', 'bodies', 'body', 'bogus', 'boiled', 'boils', 'bolsters', 'bomb', 'bombarded', 'bombs', 'bond', 'bone']

Similitud coseno entre primeros documentos (BoW):
[[1.    0.    0.    0.    0.   ]
 [0.    1.    0.    0.    0.221]
 [0.    0.    1.    0.053 0.   ]
 [0.    0.    0.053 1.    0.26 ]
 [0.    0.221 0.    0.26  1.   ]]


# TF-IDF

In [15]:

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(train_corpus)

print("\n==== TF-IDF ====")
print(f"Dimensiones de la matriz TF-IDF: {tfidf_matrix.shape}")
print(f"Ejemplo de vocabulario:          {list(tfidf_vectorizer.get_feature_names_out()[1000:1016])}")

# Similitud coseno entre los algunos documentos
similarity_tfidf = cosine_similarity(tfidf_matrix[:5])
print("\nSimilitud coseno entre primeros documentos (TF-IDF):")
print(np.round(similarity_tfidf, 3))



==== TF-IDF ====
Dimensiones de la matriz TF-IDF: (2901, 8156)
Ejemplo de vocabulario:          ['boast', 'boat', 'boats', 'bob', 'bobby', 'bodies', 'body', 'bogus', 'boiled', 'boils', 'bolsters', 'bomb', 'bombarded', 'bombs', 'bond', 'bone']

Similitud coseno entre primeros documentos (TF-IDF):
[[1.    0.    0.    0.    0.   ]
 [0.    1.    0.    0.    0.021]
 [0.    0.    1.    0.033 0.   ]
 [0.    0.    0.033 1.    0.168]
 [0.    0.021 0.    0.168 1.   ]]

Dimensiones de la matriz TF-IDF: (2901, 8156)
Ejemplo de vocabulario:          ['boast', 'boat', 'boats', 'bob', 'bobby', 'bodies', 'body', 'bogus', 'boiled', 'boils', 'bolsters', 'bomb', 'bombarded', 'bombs', 'bond', 'bone']

Similitud coseno entre primeros documentos (TF-IDF):
[[1.    0.    0.    0.    0.   ]
 [0.    1.    0.    0.    0.021]
 [0.    0.    1.    0.033 0.   ]
 [0.    0.    0.033 1.    0.168]
 [0.    0.021 0.    0.168 1.   ]]


In [16]:

# Palabras mas relevantes por documento
tfidf_array = tfidf_matrix.toarray()
feature_names = tfidf_vectorizer.get_feature_names_out()

def get_top_words(tfidf_vector, feature_names, top_n=10):
    sorted_nzs = np.argsort(tfidf_vector)[-top_n:][::-1]
    return [(feature_names[i], tfidf_vector[i]) for i in sorted_nzs]

for idx in range(3):
    print(f"\nDocumento {idx} - Palabras más relevantes (TF-IDF):")
    top_words = get_top_words(tfidf_array[idx], feature_names)
    for word, score in top_words:
        print(f"{word}: {score:.4f}")


Documento 0 - Palabras más relevantes (TF-IDF):
slogan: 0.5418
expect: 0.4225
pay: 0.4148
company: 0.3988
less: 0.3602
more: 0.2590
excluding: 0.0000
excluded: 0.0000
exclusively: 0.0000
excused: 0.0000

Documento 1 - Palabras más relevantes (TF-IDF):
child: 0.5779
handwriting: 0.4036
bigger: 0.3838
shoe: 0.3589
size: 0.3425
better: 0.2467
the: 0.2209
exercising: 0.0000
exercises: 0.0000
exercise: 0.0000

Documento 2 - Palabras más relevantes (TF-IDF):
true: 0.3983
many: 0.3766
since: 0.3689
believe: 0.3586
then: 0.3431
must: 0.2971
people: 0.2847
this: 0.2417
be: 0.2184
it: 0.2042


# Embedding no contextual con Word2Vec (fine-tuneado)

In [17]:

class EpochLogger(CallbackAny2Vec):

    def __init__(self):
        self.epoch = 0
        self.loss_previous_step = 0.0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print(f'Loss after epoch {self.epoch}: {loss}')
        else:
            print(f'Loss after epoch {self.epoch}: {loss - self.loss_previous_step}')
        self.epoch += 1
        self.loss_previous_step = loss

# Cargar modelo base preentrenado de Word2Vec (KeyedVectors)
base_kv = KeyedVectors.load_word2vec_format('./models/GoogleNews-vectors-negative300.bin.gz', binary=True, limit=400000)

# Preparar oraciones/tokenes
train_tokens = list(train["tokenized"])
test_tokens  = list(test["tokenized"])
dev_tokens   = list(dev["tokenized"])
all_sentences = train_tokens + test_tokens + dev_tokens

epoch_logger = EpochLogger()

# Crear un nuevo modelo Word2Vec para fine-tuning
model = Word2Vec(
    vector_size=base_kv.vector_size,
    window=5,
    min_count=1,
    compute_loss=True,
    sg=1,
    negative=5,
    workers=8,
)

# Construir vocabulario a partir de nuestro dataset
model.build_vocab(all_sentences)
print(f"Vocabulario del dataset: {len(model.wv)} palabras.")

# Inicializar embeddings desde el modelo preentrenado donde exista solapamiento
overlap = 0
for word, idx in list(model.wv.key_to_index.items()):
    if word in base_kv:
        model.wv[word] = base_kv[word]
        overlap += 1
print(f"Embeddings inicializados desde el modelo base: {overlap}/{len(model.wv)}")

# Asegurarnos de que todos los vectores pueden actualizarse en el fine-tuning
# Gensim usa `vectors_lockf` (o `syn0_lockf` en versiones antiguas) para controlar esto
if hasattr(model.wv, 'vectors_lockf'):
    model.wv.vectors_lockf = np.ones(len(model.wv))
else:
    try:
        model.wv.syn0_lockf = np.ones(len(model.wv))
    except Exception:
        pass

# Fine-tuning con nuestro dataset: usar model.corpus_count como total_examples
model.train(
    all_sentences,
    total_examples=model.corpus_count,
    epochs=10,
    compute_loss=True,
    callbacks=[epoch_logger],
)

# Guardar el modelo fine-tuneado
model.save("./models/word2vec_finetuned_fallacies.model")
print("Modelo fine-tuneado guardado correctamente.")

def compute_coverage(dataset, model):

    total = 0
    covered = 0

    # Si es KeyedVectors, usamos model.key_to_index
    vocab = model.key_to_index if isinstance(model, KeyedVectors) else model.wv.key_to_index

    for tokens in dataset:
        for t in tokens:
            total += 1
            if t in vocab:
                covered += 1
    return covered / total if total > 0 else 0

cov_before = compute_coverage(train_tokens, base_kv)
cov_after  = compute_coverage(train_tokens, model)
print(f"Cobertura antes del fine-tuning: {cov_before:.2%}")
print(f"Cobertura después del fine-tuning: {cov_after:.2%}")


Vocabulario del dataset: 10209 palabras.
Embeddings inicializados desde el modelo base: 8284/10209
Loss after epoch 0: 249525.859375
Embeddings inicializados desde el modelo base: 8284/10209
Loss after epoch 0: 249525.859375
Loss after epoch 1: 223474.171875
Loss after epoch 2: 212655.84375
Loss after epoch 1: 223474.171875
Loss after epoch 2: 212655.84375
Loss after epoch 3: 201186.25
Loss after epoch 4: 200409.625
Loss after epoch 3: 201186.25
Loss after epoch 4: 200409.625
Loss after epoch 5: 196513.125
Loss after epoch 6: 190578.25
Loss after epoch 5: 196513.125
Loss after epoch 6: 190578.25
Loss after epoch 7: 187137.75
Loss after epoch 8: 188847.75
Loss after epoch 7: 187137.75
Loss after epoch 8: 188847.75
Loss after epoch 9: 185978.875
Modelo fine-tuneado guardado correctamente.
Cobertura antes del fine-tuning: 76.04%
Cobertura después del fine-tuning: 100.00%
Loss after epoch 9: 185978.875
Modelo fine-tuneado guardado correctamente.
Cobertura antes del fine-tuning: 76.04%
Cobe

# Embedding contextual con BERT

In [18]:

# Cargar modelo y tokenizer BERT preentrenado
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

def get_bert_cls_embeddings(sentences, batch_size=16, max_length=128, device='cpu'):

    # Devuelve la representación del token [CLS] para cada texto (tensor en CPU)
    model.to(device)
    cls_embeddings = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        batch_texts = [" ".join(s) for s in batch]
        encoded = tokenizer(
            batch_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)
            # Tomamos la representación del token [CLS] (posición 0) del último hidden state
            batch_cls = outputs.last_hidden_state[:, 0, :]
            for emb in batch_cls:
                # Movemos a CPU para facilitar cálculo posterior (por ejemplo, similitud en CPU)
                cls_embeddings.append(emb.cpu())

    return cls_embeddings

# Generar embeddings CLS para train/test/dev
train_sentences = list(train["tokenized"])
test_sentences  = list(test["tokenized"])
dev_sentences   = list(dev["tokenized"])

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_embeddings = get_bert_cls_embeddings(train_sentences, device=device)
test_embeddings  = get_bert_cls_embeddings(test_sentences, device=device)
dev_embeddings   = get_bert_cls_embeddings(dev_sentences, device=device)

print(f"Ejemplo: embedding CLS train[0] shape: {train_embeddings[0].shape}")  # 768-d

# Ejemplo de similitud coseno (ambos tensores en CPU)
sim = F.cosine_similarity(train_embeddings[0], train_embeddings[1], dim=0)
print(f"Similitud coseno entre train[0] y train[1]: {sim.item():.4f}")


c:\Users\marco\Documents\Deusto\procesamiento_del_lenguaje_natural\fallacy-classification\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Ejemplo: embedding CLS train[0] shape: torch.Size([768])
Similitud coseno entre train[0] y train[1]: 0.8243
